# 12 - Phase 2 Discussion

Use this notebook to turn the Phase 2 results into paper text. It collects the main claims, evidence checks, limitations, and final interpretation.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np

from src.utils.io import DIRS

## Research Questions

1. How much harder is Door + Key than Phase 1 plain navigation under sparse reward?
2. Which reward formulation best supports the key -> door -> goal sequence?
3. Do DQN, DDQN, and PPO fail at different subgoals?

## Expected Interpretation Pattern

- Sparse reward should be the weakest condition because the full key -> door -> goal sequence is rarely discovered by chance.
- Subgoal reward should improve key pickup and door opening rates directly.
- Potential shaping should improve exploration without changing the optimal policy, but may be weaker than explicit subgoal bonuses for value-based methods.
- DDQN should reduce harmful overestimation compared with DQN.
- PPO may propagate delayed credit more effectively through full trajectories, especially when the sparse reward signal appears occasionally.

## Load Summary Files

In [2]:
MODE = "quick"  # use "quick" for current saved run; change to "full" for final writeup

summaries = []
for path in sorted(DIRS["logs"].glob(f"p2_{MODE}_*_summary.json")):
    summaries.append(json.loads(path.read_text(encoding="utf-8")))

print("loaded summaries:", len(summaries))
for s in summaries[:5]:
    print(s.get("run_id"), "success=", s.get("success_rate"), "key=", s.get("key_pickup_rate"), "door=", s.get("door_opening_rate"))

loaded summaries: 5
p2_quick_ddqn_potential_small_medium_seed0 success= 0.0 key= 0.25 door= 0.03
p2_quick_ddqn_sparse_small_medium_seed0 success= 0.29 key= 0.72 door= 0.42
p2_quick_ddqn_subgoal_small_medium_seed0 success= 0.77 key= 0.99 door= 0.91
p2_quick_dqn_subgoal_small_medium_seed0 success= 0.93 key= 0.94 door= 0.94
p2_quick_ppo_subgoal_small_medium_seed0 success= 0.0 key= 0.0 door= 0.0


## Claim Builder

In [3]:
def group_summary(summaries, algo=None, reward=None):
    out = []
    for s in summaries:
        if algo is not None and s.get("algo") != algo:
            continue
        if reward is not None and s.get("reward") != reward:
            continue
        out.append(s)
    return out

def mean_field(rows, field):
    vals = [r.get(field) for r in rows if r.get(field) is not None]
    return float(np.mean(vals)) if vals else None

for algo in ["dqn", "ddqn", "ppo"]:
    for reward in ["sparse", "subgoal", "potential"]:
        rows = group_summary(summaries, algo=algo, reward=reward)
        if not rows:
            continue
        print(algo, reward, {
            "success": mean_field(rows, "success_rate"),
            "key": mean_field(rows, "key_pickup_rate"),
            "door": mean_field(rows, "door_opening_rate"),
        })

dqn subgoal {'success': 0.93, 'key': 0.94, 'door': 0.94}
ddqn sparse {'success': 0.29, 'key': 0.72, 'door': 0.42}
ddqn subgoal {'success': 0.77, 'key': 0.99, 'door': 0.91}
ddqn potential {'success': 0.0, 'key': 0.25, 'door': 0.03}
ppo subgoal {'success': 0.0, 'key': 0.0, 'door': 0.0}


## Draft Result Paragraphs

### RQ1 - Difficulty Increase

Under sparse reward, the Door + Key environment introduces a longer credit-assignment chain than Phase 1. The agent must first discover the key, then return to the door, then reach the goal. Because the door is a mandatory chokepoint, bypass policies are impossible by construction.

### RQ2 - Reward Ablation

The subgoal reward condition directly rewards the two hidden milestones: key pickup and door opening. This should increase partial completion rates even before success rate fully converges. Potential shaping provides denser geometric feedback while preserving the optimal policy, making it useful as a theory-safe shaping condition.

### RQ3 - Algorithm Differences

DQN and DDQN share the same architecture and replay setting, so differences isolate the Bellman target change. PPO differs more substantially because it uses on-policy rollouts and advantage estimation, which can propagate credit through completed trajectories.

## Limitations

- The maze layout is fixed for fair comparison, so conclusions should be tested on additional maze seeds later.
- PPO and DQN/DDQN differ in both optimisation style and data budget, so direct comparisons should be interpreted carefully.
- Potential shaping depends on BFS distance tables, which are available because this is a fully observable grid with static walls.
- The key and door interactions are automatic; adding explicit pickup/use actions would create a harder action-space variant.

## Final Checklist

- [ ] Confirm every full run uses the same `MAZE_SEED = 42`.
- [ ] Confirm the door-bypass assertion passes in notebook 07.
- [ ] Confirm each DQN/DDQN/PPO x reward x seed run produced metrics or a logged explanation.
- [ ] Include V7-V10 in the final report.
- [ ] Report key pickup and door opening rates, not only final success rate.
- [ ] State whether actual results matched or contradicted the expected pattern.